In [ ]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"  # 禁止联网，只用本地 cache

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

# 本地数据集：repo_id 为子目录名，root 为父目录
dataset = LeRobotDataset(
    repo_id='/vla/.data/test',
    video_backend='torchcodec',
    )
dataset

In [ ]:
# import torch
# from lerobot.datasets.lerobot_dataset import LeRobotDataset
# from lerobot.policies.factory import make_pre_post_processors

# # Swap this import per-policy
# from lerobot.policies.pi05 import PI05Policy

# # load a policy
# model_id = "/vla/.models/lerobot-pi05_base"  # <- swap checkpoint
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# policy = PI05Policy.from_pretrained(model_id).to(device).eval()

# preprocess, postprocess = make_pre_post_processors(
#     policy.config,
#     model_id,
#     preprocessor_overrides={"device_processor": {"device": str(device)}},
# )

In [ ]:
import torch
from lerobot.utils.import_utils import register_third_party_plugins
from lerobot.policies.factory import make_policy, make_pre_post_processors
from lerobot_policy_my_policy import MyPolicyConfig

register_third_party_plugins()  # 必须：注册 lerobot_policy_my_policy 插件

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 方式 A：从零初始化（还没有训练好的 checkpoint）
config = MyPolicyConfig(device=str(device))
policy = make_policy(config, ds_meta=dataset.meta).eval()
policy

In [ ]:
preprocess, postprocess = make_pre_post_processors(
    policy.config,
    dataset_stats=dataset.meta.stats,
    preprocessor_overrides={"device_processor": {"device": str(device)}},
)

print("policy type:", policy.config.type)
print("input features:", list(policy.config.input_features.keys()))

### 推理验证

In [ ]:
batch = preprocess(dataset[0])
def to_device(batch):
    return {
        k: v.to(device, non_blocking=True) if isinstance(v, torch.Tensor) else v
        for k, v in batch.items()
    }
batch = to_device(batch)

In [ ]:
with torch.inference_mode():
    pred_action_raw = policy.select_action(batch)
pred_action_raw

### 前向传播

In [ ]:
with torch.inference_mode(): # 不进行反向传播
    loss, output_dict = policy.forward(batch)
print("loss:", loss.item() if hasattr(loss, "item") else loss)
print("output keys:", output_dict.keys() if isinstance(output_dict, dict) else type(output_dict))